<a href="https://colab.research.google.com/github/maryanamelnyk-png/Pandas-DataFrame/blob/main/Module11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#імпорт датафрейму applications.csv
import pandas as pd
file_id = "1XaRWDDRiw2sSjEorwZ5Pi-ebpJqr1Mk4"

url = f"https://drive.google.com/uc?id={file_id}"
df = pd.read_csv(url)
df.head()

,Applied at,Amount,Age,Gender,Industry,Marital status,External Rating,Education level,Location,applicant_id
0,11.30.2022 10:26:37,12000.0,29,Чоловік,Blockchain,Other,8.0,"Вища (бакалавр, спеціаліст, магістр)",Івано-Франківськ чи область,99e7b0dc6cc05dd334d8f38dc26ce9b3
1,11.30.2022 10:26:39,NaN,36,Чоловік,Public services / Government,Single,3.0,"Вища (бакалавр, спеціаліст, магістр)",NaN,63dfcf8e6904186650d6814279fbe42f
2,11.30.2022 10:26:58,7500.0,34,Чоловік,Adtech / Advertising,Single,4.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,10dbafaeb46c09e96b6987c03bbb3498
3,11.30.2022 10:27:31,1500.0,23,Жінка,Telecom,Single,0.0,"Вища (бакалавр, спеціаліст, магістр)",Львів чи область,5847ac62cc9eac5e323c2517dcc91ad1
4,11.30.2022 10:27:34,8400.0,33,Жінка,Automotive,Single,6.0,"Вища (бакалавр, спеціаліст, магістр)",Житомир чи область,5d21f3795b50de8e8f8f8d5f48b754f3


In [ ]:
df = df.drop_duplicates(subset="applicant_id")   #прибирає дублікати в applicant_id

In [ ]:
df['External Rating'] = df['External Rating'].fillna(0)  #заповнює пропуски у External Rating нулями

In [ ]:
df['Education level'] = df['Education level'].fillna("Середня")   #заповнює пропуски у Education level значенням "Середня"

In [ ]:
#імпорт industries.csv
import pandas as pd
file_id = "1dC1GhCGP8wx66r6C-kmeSCT0RlTH3s37"

url = f"https://drive.google.com/uc?id={file_id}"
industries = pd.read_csv(url)
industries.head()

,Industry,Score
0,Blockchain,0
1,Public services / Government,20
2,Adtech / Advertising,10
3,Telecom,15
4,Automotive,15


In [ ]:
df = df.merge(industries[['Industry', 'Score']],on='Industry',how='left')  #обєднання таблиць
df['Score']=df['Score'].fillna(0)

In [ ]:
df['Applied at'] = pd.to_datetime(df['Applied at'], format='mixed', dayfirst=True, errors='coerce')    #переведення дати у формат datetime

Поетапне створення допоміжних критеріїв для рейтингу:

In [ ]:
df['age_points'] = ((df['Age'] >= 35) & (df['Age'] <= 55)) * 20    #1 Якщо вік заявника між 35 та 55, до рейтингу додається 20 балів

In [ ]:
df['weekday'] = df['Applied at'].dt.weekday               #2 Якщо заявка була подана не у вихідні, то до рейтингу додається 20 балів
df['weekday_points'] = (df['weekday'] < 5) * 20

In [ ]:
df['married_points'] = (df['Marital status'] == 'Married') * 20            #3 Якщо заявник одружений, до рейтингу додається 20 балів

In [ ]:
df['location_points'] = df['Location'].str.contains("Київ", case=False) * 10          #4 Якщо заявник знаходиться в Києві чи області, до рейтингу додається 10 балів

In [ ]:
df['industry_points'] = df['Score']           #5 Значення 'Score' з таблиці industries.csv також додається до заявки (і складає від 0 до 20 балів)

In [ ]:
df['ext_high_points'] = (df['External Rating'] >= 7) * 20          #6 Якщо 'External Rating' більше чи дорівнює 7, до рейтингу додається 20 балів

In [ ]:
df['ext_low_points'] = (df['External Rating'] <= 2) * -20          #7 Якщо 'External Rating' менше чи дорівнює 2, з рейтингу віднімається 20 балів

In [ ]:
#створення підсумкового балу за зазначеними вище критеріями
df['rating_raw'] = (
    df['age_points']
    + df['weekday_points']
    + df['married_points']
    + df['location_points']
    + df['industry_points']
    + df['ext_high_points']
    + df['ext_low_points']
)

In [ ]:
df.loc[(df['Amount'].isna()) | (df['External Rating'] == 0), 'rating_raw'] = 0         #Рейтинг дорівнює нулю, якщо відсутнє значення 'Amount' або якщо 'External Rating' дорівнює нулю.

In [ ]:
df_clean = df[df['rating_raw'] > 0]                #залишаються лише заявки з рейтингом вище 0

In [ ]:
df_clean['week'] = df_clean['Applied at'].dt.isocalendar().week            #додавання колонки "week" для групування

/tmp/ipython-input-2202451525.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['week'] = df_clean['Applied at'].dt.isocalendar().week            #додавання колонки "week" для групування


In [ ]:
weekly = df_clean.groupby('week')['rating_raw'].mean().reset_index().sort_values('week')          #групування за тижнями
weekly

,week,rating_raw
0,2,57.017974
1,5,54.111111
2,6,36.899642
3,9,56.185567
4,10,35.289079
5,13,33.565891
6,15,53.651163
7,18,54.490741
8,19,50.318108
9,22,53.219697


In [ ]:
num_weeks = weekly.shape[0]
print("Кількість тижнів з прийнятими заявками:", num_weeks)

Кількість тижнів з прийнятими заявками: 23


In [ ]:
df.isna().sum()

,0
Applied at,0
Amount,33
Age,0
Gender,0
Industry,0
Marital status,0
External Rating,0
Education level,0
Location,1772
applicant_id,0


In [ ]:
df[df['rating_raw']==0]

,Applied at,Amount,Age,Gender,Industry,Marital status,External Rating,Education level,Location,applicant_id,Score,age_points,weekday,weekday_points,married_points,location_points,industry_points,ext_high_points,ext_low_points,rating_raw
1,2022-11-30 10:26:39,NaN,36,Чоловік,Public services / Government,Single,3.0,"Вища (бакалавр, спеціаліст, магістр)",NaN,63dfcf8e6904186650d6814279fbe42f,20,20,2,20,0,NaN,20,0,0,0
3,2022-11-30 10:27:31,1500.0,23,Жінка,Telecom,Single,0.0,"Вища (бакалавр, спеціаліст, магістр)",Львів чи область,5847ac62cc9eac5e323c2517dcc91ad1,15,0,2,20,0,0,15,0,-20,0
9,2022-11-30 10:28:00,NaN,17,Чоловік,Інша,Married,2.0,Ще студент вишу,NaN,9c7a640dccb37d9f2164bac1deb0edfa,10,0,2,20,20,NaN,10,0,-20,0
11,2022-11-30 10:28:21,19800.0,29,Чоловік,Edtech / Education,Single,0.0,"Вища (бакалавр, спеціаліст, магістр)",Львів чи область,9db8b1c7ed0fc6e3fe79a41ac3468442,15,0,2,20,0,0,15,0,-20,0
17,2022-11-30 10:30:00,NaN,22,Чоловік,Blockchain,Single,2.0,Ще студент вишу,NaN,bef0a5ba4df413cb8e1e3edeaf1f7de3,0,0,2,20,0,NaN,0,0,-20,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13233,2023-07-01 09:10:00,5700.0,24,Жінка,E-commerce,Married,0.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,43c738c2ee2d13a99753b0ed71397555,15,0,5,0,20,10,15,0,-20,0
13236,2023-07-01 11:34:00,1020.0,39,Чоловік,Adtech / Advertising,Married,0.0,"Вища (бакалавр, спеціаліст, магістр)",Дніпро чи область,059b8410f2792acfd97697db96dbd136,10,20,5,0,20,0,10,0,-20,0
13238,2023-07-01 13:57:00,4800.0,22,Чоловік,Інша,Married,0.0,"Вища (бакалавр, спеціаліст, магістр)",Львів чи область,eae8324e053b814d86dc50a290be37e6,10,0,5,0,20,0,10,0,-20,0
13266,2023-08-01 22:25:00,7500.0,26,Жінка,Medtech / Healthcare,Married,0.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,570929ecff6eab12338c89b311eefdb8,15,0,1,20,20,10,15,0,-20,0


In [ ]:
df_clean.head(20)

,Applied at,Amount,Age,Gender,Industry,Marital status,External Rating,Education level,Location,applicant_id,...,age_points,weekday,weekday_points,married_points,location_points,industry_points,ext_high_points,ext_low_points,rating_raw,week
0,2022-11-30 10:26:37,12000.0,29,Чоловік,Blockchain,Other,8.0,"Вища (бакалавр, спеціаліст, магістр)",Івано-Франківськ чи область,99e7b0dc6cc05dd334d8f38dc26ce9b3,...,0,2,20,0,0,0,20,0,40,48
2,2022-11-30 10:26:58,7500.0,34,Чоловік,Adtech / Advertising,Single,4.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,10dbafaeb46c09e96b6987c03bbb3498,...,0,2,20,0,10,10,0,0,40,48
4,2022-11-30 10:27:34,8400.0,33,Жінка,Automotive,Single,6.0,"Вища (бакалавр, спеціаліст, магістр)",Житомир чи область,5d21f3795b50de8e8f8f8d5f48b754f3,...,0,2,20,0,0,15,0,0,35,48
5,2022-11-30 10:27:38,16500.0,31,Чоловік,E-commerce,Single,8.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,f720bf9c5c4c3e10a8568c1699847696,...,0,2,20,0,10,15,20,0,65,48
6,2022-11-30 10:27:42,4200.0,30,Чоловік,Media,Married,1.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,0aaf59fb3ef90f50ccd4800312e5c271,...,0,2,20,20,10,5,0,-20,35,48
7,2022-11-30 10:27:44,3600.0,24,Чоловік,E-commerce,Married,1.0,"Вища (бакалавр, спеціаліст, магістр)",Львів чи область,aa54d6e672e4233febadbafdfd048327,...,0,2,20,20,0,15,0,-20,35,48
8,2022-11-30 10:28:00,1950.0,27,Чоловік,Automotive,Married,1.0,"Вища (бакалавр, спеціаліст, магістр)",Львів чи область,2eed2883cd1673b21b2ce89d1115c245,...,0,2,20,20,0,15,0,-20,35,48
10,2022-11-30 10:28:03,18000.0,25,Чоловік,Dating,Single,5.0,"Вища (бакалавр, спеціаліст, магістр)",Дніпро чи область,f8138219d5a95649cc85bdabeb3732ca,...,0,2,20,0,0,5,0,0,25,48
12,2022-11-30 10:28:27,2550.0,20,Чоловік,Legal,Married,1.0,Ще студент вишу,Івано-Франківськ чи область,0c792f52f2bf30cdfee53b24b912678a,...,0,2,20,20,0,20,0,-20,40,48
13,2022-11-30 10:28:28,18000.0,33,Чоловік,Big Data,Single,10.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,28165b1a85ee2e48782cada5f2a660b4,...,0,2,20,0,10,20,20,0,70,48
